# 05 — Evaluación: baseline contra intermedio contra final

Este notebook produce **la tabla** del §5 del enunciado: el mismo agente en
tres configuraciones, evaluado sobre los mismos conjuntos de preguntas, con
las métricas de calidad y las de coste una al lado de la otra.

## Qué se compara, y por qué esas tres configuraciones

El enunciado pide «baseline contra intermedio contra final». La tentación es
elegir tres versiones cualesquiera que vayan mejorando; el problema de
hacerlo así es que la tabla resultante no permite atribuir la mejora a nada
concreto. Si entre el baseline y el final cambian a la vez el retrieval, el
prompt y los guardrails, la tabla dice «mejoró» y no dice **qué** mejoró.

Por eso las tres configuraciones se diferencian de forma **acumulativa y en
un solo eje cada vez**:

| Sistema | Retrieval de `search_filings` | Guardrail numérico | Qué añade respecto al anterior |
| --- | --- | --- | --- |
| `baseline` | denso plano, `k=5`, sin filtros | no | — |
| `filtros` | denso + filtros de metadatos | no | **solo** el filtro por ticker/ejercicio/item |
| `final` | reescritura + híbrido BM25/RRF + reranking | **sí** | reescritura, fusión léxica, reranking y verificación contra XBRL |

Todo lo demás —modelo, temperatura, prompt de sistema, esquema de salida,
límites de llamadas, `k`— es idéntico en los tres. Es la única forma de que
la diferencia entre dos filas sea atribuible a lo que se movió.

## Sobre los conjuntos de evaluación

Se evalúa sobre **tres** conjuntos, y cada uno responde a una pregunta
distinta:

1. **`golden_set.jsonl`, el oficial.** Es el que no hemos escrito nosotros,
   así que es el único que mide sin sesgo de autoría. Es la fila que cuenta
   para comparar con otros grupos.
2. **`golden_set_propio.jsonl`, las 20 nuestras.** Cubre los seis emisores,
   los dos ejercicios y los cuatro items, cosa que el oficial no hace. Sirve
   para ver si el sistema se cae en alguna combinación concreta.
3. **`golden_set_ausencias.jsonl`.** Siete preguntas cuya respuesta correcta
   es «ese dato no está en el corpus». No entra en la tabla principal porque
   su criterio de acierto es distinto —`fuente == "ninguna"` y `cifra` nula—
   y mezclarla con las demás inflaría o hundiría la métrica según cuántas
   hubiera. Va en sección propia.

## Coste de ejecutar este notebook

Tres sistemas × (20 + 20 + 7) preguntas = 141 invocaciones del agente, cada
una con varias llamadas al modelo. Con `gpt-5-mini` y los precios de
`agente/config.py`, el orden de magnitud es de unos pocos céntimos en total,
pero **tarda**: entre 15 y 40 minutos según la latencia de la API. Los
resultados crudos se guardan en `resultados/`, así que el notebook 06 —el
del informe— los lee de disco y no vuelve a pagar nada.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
for carpeta in (RAIZ, RAIZ / "modulos"):
    if str(carpeta) not in sys.path:
        sys.path.insert(0, str(carpeta))

import json
import time

import pandas as pd

from agente import config, interfaz, retrieval
from agente.agente import DESCRIPCION_PERFILES

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

CONJUNTOS = {
    "oficial": config.RUTA_GOLDEN_OFICIAL,
    "propio": config.RUTA_GOLDEN_PROPIO,
    "ausencias": config.RUTA_GOLDEN_AUSENCIAS,
}
SISTEMAS = ["baseline", "filtros", "final"]

config.DIR_RESULTADOS.mkdir(exist_ok=True)

for nombre, ruta in CONJUNTOS.items():
    existe = "OK" if Path(ruta).is_file() else "FALTA"
    n = len(interfaz.cargar_golden(ruta)) if Path(ruta).is_file() else 0
    print(f"  {nombre:10s} {existe:5s} {n:2d} preguntas   {ruta.name}")

print()
for sistema in SISTEMAS:
    print(f"  {sistema:9s} {DESCRIPCION_PERFILES[sistema]}")

HAY_MODELO = config.hay_modelo()
print(f"\nModelo disponible: {HAY_MODELO}")
if HAY_MODELO:
    print(json.dumps(config.resumen_configuracion(), indent=2, ensure_ascii=False))
else:
    print(config.motivo_sin_modelo())

  oficial    OK    20 preguntas   golden_set.jsonl
  propio     OK    20 preguntas   golden_set_propio.jsonl
  ausencias  OK     7 preguntas   golden_set_ausencias.jsonl

  baseline  denso plano, sin filtros, sin guardrail numérico
  filtros   denso + filtros de metadatos, sin guardrail numérico
  final     reescritura + híbrido BM25/RRF + reranking, con guardrail numérico

Modelo disponible: True
{
  "modelo": "gpt-5-mini",
  "temperatura": 1,
  "modelo_embeddings": "BAAI/bge-small-en-v1.5",
  "k": 5,
  "tolerancia_cifra": 0.01,
  "limite_llamadas_herramienta": 8,
  "limite_llamadas_modelo": 10,
  "precio_entrada_usd_por_millon": 0.25,
  "precio_salida_usd_por_millon": 2.0,
  "fecha_precios": "2026-09-21"
}


## La configuración exacta, registrada junto a los resultados

`config.resumen_configuracion()` se imprime arriba y se guarda con cada
fichero de resultados. No es burocracia: un número sin la configuración que
lo produjo no es reproducible, y el modelo concreto que la cuenta tenga
disponible el día de la ejecución es precisamente lo que no controlamos.
Si el informe dice «acierto del 85 %» y no dice con qué modelo, a qué
temperatura, con qué `k` y con qué tolerancia, la cifra no se puede
contrastar ni repetir.

El precio también va fechado. Los tokens los reporta la API; los dólares los
ponemos nosotros multiplicando por una tabla que caduca.

In [2]:
EJECUTAR = HAY_MODELO  # si no hay modelo, solo se puede recalcular lo ya hecho
FORZAR = False         # ponlo en True para reejecutar aunque haya crudos

RESULTADOS: dict[tuple[str, str], pd.DataFrame] = {}


def ruta_de(sistema: str, conjunto: str) -> Path:
    return config.DIR_RESULTADOS / f"eval_{conjunto}_{sistema}"


def evaluar_o_leer(sistema: str, conjunto: str) -> pd.DataFrame:
    """Recalcula desde los crudos si ya están en disco; si no, evalúa.

    Esta función es la razón de que revisar el informe no cueste dinero, y
    merece la pena explicar por qué está escrita así y no de la forma obvia.

    Lo obvio sería releer el CSV. El problema de releer el CSV es que las
    columnas de evaluación quedan **congeladas** con la definición que tenían
    los evaluadores el día que se ejecutó. Nos pasó: la primera pasada completa
    reveló que el acierto extractivo no distinguía entre el mejor y el peor de
    los sistemas, hubo que añadir `cita_fundamentada`, y con un CSV congelado la
    única forma de actualizar la tabla habría sido volver a pagar las 141
    invocaciones. Esa fricción empuja a no corregir un evaluador que se ha
    quedado corto, que es exactamente lo contrario de lo que debe pasar.

    Lo que se relee, entonces, es el **`.jsonl` con las respuestas crudas**, y
    los evaluadores se vuelven a aplicar con el código de hoy. La separación es
    limpia: invocar al agente cuesta dinero y minutos y su resultado es un
    hecho; puntuar ese resultado es gratis y su definición puede cambiar.

    `FORZAR = True` reejecuta aunque haya crudos, que es lo que hay que hacer
    cuando lo que cambia es el agente y no el evaluador.
    """
    destino = ruta_de(sistema, conjunto)
    crudos = destino.with_suffix(".jsonl")

    if crudos.is_file() and not FORZAR:
        n_esperadas = len(interfaz.cargar_golden(CONJUNTOS[conjunto]))
        n_guardadas = sum(1 for linea in crudos.open(encoding="utf-8") if linea.strip())
        if n_guardadas >= n_esperadas:
            tabla = interfaz.recalcular(crudos, CONJUNTOS[conjunto], perfil=sistema)
            tabla.to_csv(destino.with_suffix(".csv"), index=False, encoding="utf-8")
            print(f"  {conjunto:9s} {sistema:9s} recalculado desde {crudos.name} "
                  f"({len(tabla)} preguntas, sin coste)")
            return tabla
        print(f"  {conjunto:9s} {sistema:9s} crudos incompletos "
              f"({n_guardadas}/{n_esperadas}): se reejecuta")

    if not EJECUTAR:
        print(f"  {conjunto:9s} {sistema:9s} no hay crudos y no hay modelo: "
              f"{config.motivo_sin_modelo()}")
        return pd.DataFrame()

    print(f"\n=== {conjunto} · {sistema} "
          f"({DESCRIPCION_PERFILES[sistema]}) ===")
    return interfaz.evaluar(
        CONJUNTOS[conjunto], perfil=sistema, guardar_en=destino
    )


comienzo = time.perf_counter()
for conjunto in CONJUNTOS:
    for sistema in SISTEMAS:
        RESULTADOS[(conjunto, sistema)] = evaluar_o_leer(sistema, conjunto)
print(f"\nTiempo total: {(time.perf_counter() - comienzo) / 60:.1f} min")

  oficial   baseline  recalculado desde eval_oficial_baseline.jsonl (20 preguntas, sin coste)
  oficial   filtros   recalculado desde eval_oficial_filtros.jsonl (20 preguntas, sin coste)
  oficial   final     recalculado desde eval_oficial_final.jsonl (20 preguntas, sin coste)
  propio    baseline  recalculado desde eval_propio_baseline.jsonl (20 preguntas, sin coste)
  propio    filtros   recalculado desde eval_propio_filtros.jsonl (20 preguntas, sin coste)
  propio    final     recalculado desde eval_propio_final.jsonl (20 preguntas, sin coste)
  ausencias baseline  recalculado desde eval_ausencias_baseline.jsonl (7 preguntas, sin coste)
  ausencias filtros   recalculado desde eval_ausencias_filtros.jsonl (7 preguntas, sin coste)
  ausencias final     recalculado desde eval_ausencias_final.jsonl (7 preguntas, sin coste)

Tiempo total: 0.0 min


## `recall@5` medido sobre la trayectoria real, no sobre el golden set

El `recall@5` del notebook 04 se calcula pasándole al retriever los filtros
**perfectos** que trae el golden set: el ticker, el ejercicio y el item
correctos. Ese número es un techo, y es el que hay que usar para comparar
estrategias de retrieval entre sí, porque aísla la estrategia de los errores
del agente.

Aquí interesa el otro número: el `recall@5` del sistema **completo**, donde
los filtros los infiere el agente a partir de la pregunta. Es sistemáticamente
más bajo, y la diferencia entre los dos mide exactamente una cosa: cuánto
pierde el sistema por equivocarse al enrutar.

Se calcula reproduciendo la búsqueda con la misma estrategia que usó cada
perfil, sin filtros dados. No cuesta ninguna llamada al modelo salvo la
reescritura, que está cacheada de la evaluación anterior.

In [3]:
ESTRATEGIA_DE = {"baseline": "denso_plano", "filtros": "filtros", "final": "final"}


def recall_del_sistema(conjunto: str, sistema: str) -> float | None:
    """recall@5 sin darle al retriever los filtros del golden set."""
    items = [g for g in interfaz.cargar_golden(CONJUNTOS[conjunto])
             if g.get("ancla_texto")]
    if not items:
        return None

    aciertos = 0
    for g in items:
        fragmentos = retrieval.buscar(
            g["pregunta"], estrategia=ESTRATEGIA_DE[sistema],
            k=config.K_POR_DEFECTO,
        )
        aciertos += bool(retrieval.acierta(g, fragmentos))
    return aciertos / len(items)


RECALL: dict[tuple[str, str], float | None] = {}
for conjunto in ("oficial", "propio"):
    for sistema in SISTEMAS:
        RECALL[(conjunto, sistema)] = recall_del_sistema(conjunto, sistema)
        valor = RECALL[(conjunto, sistema)]
        print(f"  {conjunto:8s} {sistema:9s} recall@5 sin filtros dados = "
              f"{'n/a' if valor is None else f'{valor:.3f}'}")

  oficial  baseline  recall@5 sin filtros dados = 0.308
  oficial  filtros   recall@5 sin filtros dados = 0.308


  oficial  final     recall@5 sin filtros dados = 0.538


  propio   baseline  recall@5 sin filtros dados = 0.231


  propio   filtros   recall@5 sin filtros dados = 0.231


  propio   final     recall@5 sin filtros dados = 0.615


## La tabla comparativa

Una fila por sistema y conjunto. Las columnas están agrupadas en tres
bloques, y el orden no es casual: primero **si acierta**, después **si se
puede comprobar que acierta**, y al final **cuánto cuesta**.

- **Calidad**: acierto global y acierto desglosado por familia. El desglose
  importa porque las tres familias se rompen por sitios distintos, y una
  media global puede esconder que las comparativas van mal.
- **Verificabilidad**: `cita`, `cifra` y `trayectoria`, los tres evaluadores
  del §4, más `fundamentada` y el `recall@5` del sistema completo. Un acierto
  sin cita verificada no es un acierto que se pueda defender.

Sobre la columna `fundamentada`, que no está en el enunciado y que hemos
añadido después de medir: `cita_correcta` comprueba que la cita sea **real**
—que el `chunk_id` exista, que el fragmento sea del documento correcto y que
el texto esté de verdad ahí—, y eso es lo que pide el §4. Lo que no puede
comprobar es si ese fragmento real tiene algo que ver con la pregunta.

El problema no es teórico: en la primera ejecución completa, el sistema
`baseline` —el que peor recupera de los tres— acertaba el 100 % de las
preguntas extractivas, porque con cuarenta fragmentos por sección siempre
encuentra alguno real de la compañía correcta que citar. Una métrica que
puntúa igual al mejor y al peor de los sistemas no está midiendo el sistema.

`fundamentada` cierra ese hueco con el dato que el golden set ya trae: el
fragmento citado tiene que **solapar** con el tramo `[ancla_inicio,
ancla_fin)` donde vive la frase que responde. Se exige solape y no
contención porque el troceador puede partir un ancla entre dos fragmentos, y
en ese caso los dos son citas legítimas.
- **Coste**: dólares por pregunta, latencia y llamadas a herramienta. Es la
  columna que convierte «el final es mejor» en «el final es mejor y cuesta
  N veces más», que es la afirmación que de verdad se puede defender.

El mejor valor de cada columna va marcado con `*`. En las columnas de coste
y latencia, «mejor» es el valor **más bajo**, así que ojo al leer: un
asterisco en la columna de coste no es un elogio al sistema, es el aviso de
que el barato probablemente sea también el peor.

In [4]:
COLUMNAS_MENOR_ES_MEJOR = {"coste medio ($)", "latencia (s)", "llamadas/preg"}


def fila_resumen(conjunto: str, sistema: str) -> dict:
    tabla = RESULTADOS[(conjunto, sistema)]
    if tabla.empty:
        return {"conjunto": conjunto, "sistema": sistema}
    r = interfaz.resumir(tabla, sistema)
    return {
        "conjunto": conjunto,
        "sistema": sistema,
        "n": r["n"],
        "acierto": r["acierto"],
        "extractiva": r["acierto_extractiva"],
        "numérica": r["acierto_numerica"],
        "comparativa": r["acierto_comparativa"],
        "cita": r["cita"],
        "cifra": r["cifra"],
        "trayectoria": r["trayectoria"],
        "fundamentada": r.get("fundamentada"),
        "recall@5": RECALL.get((conjunto, sistema)),
        "coste medio ($)": r["coste_medio_usd"],
        "latencia (s)": r["latencia_media_s"],
        "llamadas/preg": r["llamadas_medias"],
        "guardrail": r["guardrail_saltó"],
        "errores": r["errores"],
    }


def resaltar(tabla: pd.DataFrame, agrupar_por: str = "conjunto") -> pd.DataFrame:
    """Devuelve la tabla formateada, con un `*` en el mejor valor por columna.

    El resaltado se hace DENTRO de cada conjunto de preguntas, no sobre toda
    la tabla: comparar el acierto del baseline en el golden propio con el del
    final en el oficial no significa nada, porque no son las mismas preguntas.
    """
    numericas = [c for c in tabla.columns
                 if c not in {"conjunto", "sistema", "n", "guardrail", "errores"}
                 and pd.api.types.is_numeric_dtype(tabla[c])]
    salida = tabla.copy()
    for columna in numericas:
        salida[columna] = salida[columna].astype(object)

    for columna in numericas:
        formato = ("{:.4f}" if "$" in columna
                   else "{:.1f}" if columna in {"latencia (s)", "llamadas/preg"}
                   else "{:.3f}")
        for _, indices in tabla.groupby(agrupar_por).groups.items():
            bloque = tabla.loc[indices, columna].dropna()
            if bloque.empty:
                continue
            mejor = bloque.min() if columna in COLUMNAS_MENOR_ES_MEJOR else bloque.max()
            for i in indices:
                v = tabla.loc[i, columna]
                if pd.isna(v):
                    salida.at[i, columna] = "n/a"
                else:
                    salida.at[i, columna] = (
                        formato.format(v) + ("*" if v == mejor else " ")
                    )
    return salida


comparativa = pd.DataFrame([
    fila_resumen(conjunto, sistema)
    for conjunto in ("oficial", "propio")
    for sistema in SISTEMAS
])

if comparativa.drop(columns=["conjunto", "sistema"]).notna().any().any():
    print(resaltar(comparativa).to_string(index=False))
    comparativa.to_csv(
        config.DIR_RESULTADOS / "tabla_comparativa.csv", index=False, encoding="utf-8")
    print(f"\nGuardado en {config.DIR_RESULTADOS / 'tabla_comparativa.csv'}")
else:
    print("Sin resultados que resumir: ejecuta el notebook con clave.")

conjunto  sistema  n acierto extractiva numérica comparativa   cita  cifra trayectoria fundamentada recall@5 coste medio ($) latencia (s) llamadas/preg  guardrail  errores
 oficial baseline 20  0.800*     0.833    1.000*      0.571* 0.923  1.000*      1.000*       0.692*   0.308          0.0073         16.6*          2.8           0        0
 oficial  filtros 20  0.750      0.833    1.000*      0.429  0.923  1.000*      1.000*       0.615    0.308          0.0072         17.1           2.8           0        0
 oficial    final 20  0.800*     1.000*   1.000*      0.429  1.000* 1.000*      1.000*       0.692*   0.538*         0.0072*        23.0           2.5*          0        0
  propio baseline 20  0.667      0.714    1.000*      0.000  0.818  0.909       1.000*       0.455    0.231          0.0097         20.3           3.0           0        2
  propio  filtros 20  0.750*     1.000*   1.000*      0.167  1.000* 0.923       1.000*       0.692*   0.231          0.0069*        17.9*   

## Las preguntas de ausencia, aparte

Siete preguntas cuya respuesta correcta es que el dato **no está**. Se
evalúan con un criterio propio, y conviene explicar por qué no se pueden
meter en la tabla de arriba.

El acierto de una pregunta numérica es «la cifra coincide con la del XBRL
dentro del 1 %». Una pregunta de ausencia no tiene cifra que coincidir: su
acierto es exactamente lo contrario, que el agente **no** produzca ninguna.
Si se mezclaran, el evaluador de cifras devolvería `None` para las siete y el
de acierto mediría dos cosas distintas bajo el mismo nombre.

El criterio aquí es doble y las dos partes hacen falta: `fuente == "ninguna"`
**y** `cifra is None`. Solo la primera dejaría pasar una respuesta que dice
«no consta» y a la vez rellena el campo numérico con una estimación, que es
justo el fallo que estas preguntas están puestas para detectar.

In [5]:
def acierta_ausencia(resultado_fila: pd.Series) -> bool:
    return (str(resultado_fila.get("fuente")) == "ninguna"
            and pd.isna(resultado_fila.get("cifra_afirmada")))


filas_ausencias = []
for sistema in SISTEMAS:
    tabla = RESULTADOS[("ausencias", sistema)]
    if tabla.empty:
        continue
    correctas = tabla.apply(acierta_ausencia, axis=1)
    filas_ausencias.append({
        "sistema": sistema,
        "n": len(tabla),
        "dice 'no está'": float(correctas.mean()),
        "inventa cifra": float((tabla["cifra_afirmada"].notna()).mean()),
        "coste medio ($)": float(tabla["coste_usd"].mean()),
        "latencia (s)": float(tabla["latencia_s"].mean()),
    })

if filas_ausencias:
    tabla_ausencias = pd.DataFrame(filas_ausencias)
    print(tabla_ausencias.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    tabla_ausencias.to_csv(
        config.DIR_RESULTADOS / "tabla_ausencias.csv", index=False, encoding="utf-8")

    print("\nPregunta a pregunta, con el sistema final:")
    detalle = RESULTADOS[("ausencias", "final")]
    if not detalle.empty:
        columnas = ["id", "ticker", "fuente", "cifra_afirmada", "respuesta"]
        vista = detalle[[c for c in columnas if c in detalle]].copy()
        vista["respuesta"] = vista["respuesta"].astype(str).str.slice(0, 90)
        print(vista.to_string(index=False))
else:
    print("Sin resultados de ausencias.")

 sistema  n  dice 'no está'  inventa cifra  coste medio ($)  latencia (s)
baseline  7           0.857          0.143            0.005        13.574
 filtros  7           0.857          0.143            0.005        13.996
   final  7           1.000          0.000            0.005        14.579

Pregunta a pregunta, con el sistema final:
    id ticker  fuente  cifra_afirmada                                                                                  respuesta
ga-001   AMZN ninguna             NaN No está disponible en el corpus: Amazon no reporta ni GrossProfit ni CostOfRevenue en sus 
ga-002   AMZN ninguna             NaN No está disponible en el corpus: Amazon no reportó 'ResearchAndDevelopmentExpense' como un
ga-003   META ninguna             NaN Meta no reportó el concepto 'GrossProfit' ni el 'gross margin' en su XBRL para el ejercici
ga-004   AMZN ninguna             NaN No está disponible en el corpus: Amazon no reportó el concepto 'Liabilities' en XBRL para 
ga-005   NVDA n

## Qué falló, y por qué

Una tabla de porcentajes dice qué proporción falló y no dice nada útil sobre
cómo arreglarlo. Esta sección baja al detalle de las preguntas que el sistema
final sigue fallando, con la trayectoria completa guardada en
`resultados/eval_*_final.jsonl` para poder leerla después.

La clasificación que interesa es por **dónde** se rompió, porque cada sitio
tiene un arreglo distinto:

- **No recuperó el fragmento** (`recall` falso): el problema es el retriever.
- **Recuperó y no citó** (`cita` falsa con `recall` cierto): el problema es
  el prompt o el esquema, no la búsqueda.
- **No llamó a la herramienta esperada** (`trayectoria` falsa): el problema
  es el enrutado, es decir, los docstrings.
- **Llamó bien y la cifra no cuadra** (`cifra` falsa con `trayectoria`
  cierta): pidió el concepto equivocado.

In [6]:
def diagnosticar(tabla: pd.DataFrame) -> pd.DataFrame:
    """Clasifica cada fallo por la etapa en la que se rompió."""
    if tabla.empty:
        return tabla

    def etapa(fila) -> str:
        if fila.get("error"):
            return "excepción"
        if fila.get("trayectoria") is False:
            return "enrutado (no usó la herramienta esperada)"
        if fila.get("cifra") is False:
            return "cifra (concepto equivocado o no consultado)"
        if fila.get("cita") is False:
            return "cita (no citó, o citó algo que no respalda)"
        if fila.get("acierto") is False:
            return "redacción (todo verificable, respuesta insuficiente)"
        return "correcta"

    copia = tabla.copy()
    copia["etapa"] = copia.apply(etapa, axis=1)
    return copia


for conjunto in ("oficial", "propio"):
    tabla = diagnosticar(RESULTADOS[(conjunto, "final")])
    if tabla.empty:
        continue
    print(f"\n=== {conjunto} · sistema final ===")
    print(tabla["etapa"].value_counts().to_string())
    fallos = tabla[tabla["etapa"] != "correcta"]
    if not fallos.empty:
        columnas = ["id", "familia", "ticker", "etapa", "respuesta"]
        vista = fallos[[c for c in columnas if c in fallos]].copy()
        vista["respuesta"] = vista["respuesta"].astype(str).str.slice(0, 80)
        print(vista.to_string(index=False))


=== oficial · sistema final ===
etapa
correcta                                                16
redacción (todo verificable, respuesta insuficiente)     4
    id     familia ticker                                                etapa                                                                        respuesta
of-014 comparativa   MSFT redacción (todo verificable, respuesta insuficiente) El revenue de Microsoft creció de $245,122,000,000 en FY2024 a $281,724,000,000 
of-015 comparativa   NVDA redacción (todo verificable, respuesta insuficiente) El revenue de NVIDIA creció de $60,922,000,000 en FY2024 a $130,497,000,000 en F
of-017 comparativa   AMZN redacción (todo verificable, respuesta insuficiente) El beneficio operativo aumentó de 68,593 millones USD en 2024 a 79,975 millones 
of-018 comparativa  GOOGL redacción (todo verificable, respuesta insuficiente) Los ingresos de Alphabet pasaron de $350,018,000,000 en 2024 a $402,836,000,000 

=== propio · sistema final ===
etapa
corre

## El guardrail numérico: cuántas veces intervino y si sirvió

El middleware que contrasta cada cifra contra el XBRL solo está activo en el
sistema `final`. La pregunta que hay que contestar con datos, y no con fe,
es si intervenir sirve de algo: **de las preguntas en las que saltó, ¿en
cuántas la respuesta final acabó siendo correcta?**

Hay dos formas de que un guardrail sea inútil, y las dos se ven aquí:

- **Que no salte nunca.** Entonces no está aportando nada y solo añade una
  llamada de verificación por pregunta numérica.
- **Que salte y el modelo no se corrija.** Entonces está pagando una vuelta
  extra del modelo para acabar en el mismo sitio, y lo correcto sería
  convertirlo en un rechazo duro en vez de en una sugerencia.

In [7]:
for conjunto in ("oficial", "propio", "ausencias"):
    tabla = RESULTADOS[(conjunto, "final")]
    if tabla.empty or "guardrail" not in tabla:
        continue
    salto = tabla["guardrail"].fillna(False).astype(bool)
    print(f"\n{conjunto}: saltó en {int(salto.sum())} de {len(tabla)} preguntas")
    if salto.any():
        con_salto = tabla[salto]
        aciertos = con_salto["acierto"].dropna()
        if len(aciertos):
            print(f"  de esas, acertaron {int(aciertos.sum())}/{len(aciertos)} "
                  f"({aciertos.mean():.0%})")
        sin_salto = tabla[~salto]["acierto"].dropna()
        if len(sin_salto):
            print(f"  sin intervención, acertaron {int(sin_salto.sum())}/"
                  f"{len(sin_salto)} ({sin_salto.mean():.0%})")
        print(f"  ids: {list(con_salto['id'])}")


oficial: saltó en 0 de 20 preguntas

propio: saltó en 1 de 20 preguntas
  de esas, acertaron 1/1 (100%)
  sin intervención, acertaron 15/19 (79%)
  ids: ['gp-006']

ausencias: saltó en 0 de 7 preguntas


## Coste total de la evaluación

La cifra que hay que saber antes del día 24: cuánto cuesta y cuánto tarda
pasar el sistema final por un conjunto de preguntas. El hold-out son diez
preguntas y la ventana son veinte minutos.

In [8]:
resumen_coste = []
for (conjunto, sistema), tabla in RESULTADOS.items():
    if tabla.empty:
        continue
    resumen_coste.append({
        "conjunto": conjunto,
        "sistema": sistema,
        "preguntas": len(tabla),
        "coste total ($)": float(tabla["coste_usd"].sum()),
        "$/pregunta": float(tabla["coste_usd"].mean()),
        "minutos": float(tabla["latencia_s"].sum()) / 60,
        "s/pregunta": float(tabla["latencia_s"].mean()),
    })

if resumen_coste:
    tc = pd.DataFrame(resumen_coste)
    print(tc.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print(f"\nTOTAL de este notebook: ${tc['coste total ($)'].sum():.3f} · "
          f"{tc['minutos'].sum():.1f} min")

    final_oficial = RESULTADOS[("oficial", "final")]
    if not final_oficial.empty:
        por_pregunta = final_oficial["latencia_s"].mean()
        print(f"\nProyección para el hold-out de 10 preguntas con el sistema "
              f"final: {10 * por_pregunta / 60:.1f} min, "
              f"${10 * final_oficial['coste_usd'].mean():.3f}.")
        print(f"La ventana del día 24 son 20 minutos.")
else:
    print("Sin resultados.")

 conjunto  sistema  preguntas  coste total ($)  $/pregunta  minutos  s/pregunta
  oficial baseline         20            0.145       0.007    5.528      16.584
  oficial  filtros         20            0.145       0.007    5.714      17.141
  oficial    final         20            0.136       0.007    6.698      20.095
   propio baseline         20            0.194       0.010    6.773      20.320
   propio  filtros         20            0.137       0.007    5.960      17.879
   propio    final         20            0.150       0.008    7.916      23.747
ausencias baseline          7            0.032       0.005    1.584      13.574
ausencias  filtros          7            0.033       0.005    1.633      13.996
ausencias    final          7            0.034       0.005    2.005      17.186

TOTAL de este notebook: $1.006 · 43.8 min

Proyección para el hold-out de 10 preguntas con el sistema final: 3.3 min, $0.068.
La ventana del día 24 son 20 minutos.


## Conclusiones de este notebook

Las conclusiones cuantitativas se escriben a partir de la tabla de arriba y
se recogen en el informe (notebook 06). Lo que conviene dejar dicho aquí,
porque es metodológico y no depende de los números concretos:

**La potencia estadística es baja y hay que decirlo.** Veinte preguntas por
conjunto significan que una sola pregunta vale cinco puntos porcentuales. Una
diferencia de una o dos preguntas entre dos sistemas no es distinguible del
ruido, y presentarla como una mejora sería un error de interpretación, no una
exageración retórica. Las diferencias que sí se pueden defender son las de
varias preguntas seguidas y, sobre todo, las que aparecen **en los dos
conjuntos a la vez**: que una mejora se reproduzca en el golden set oficial,
que no hemos escrito nosotros, es la mejor evidencia disponible de que no es
un artefacto de cómo redactamos nuestras preguntas.

**El coste no es una nota al pie.** El sistema final añade, por pregunta, una
llamada de reescritura, un pase de cross-encoder y —en las numéricas— una
posible vuelta extra del modelo por el guardrail. Si la mejora en acierto es
pequeña y el coste se multiplica, la decisión correcta de ingeniería puede
ser quedarse en el intermedio. La tabla está montada para poder tomar esa
decisión con datos en lugar de por inercia.

**Los resultados crudos se guardan enteros.** `resultados/eval_*.jsonl`
contiene la trayectoria completa de cada invocación. Es lo que permite
volver sobre un fallo tres días después y ver qué herramienta llamó el
agente y con qué argumentos, en lugar de tener que pagar la ejecución otra
vez para averiguarlo.